### Diversity Generator
This code generates a random Bryant-Tupper diversity by incrementally assigning random numbers to a finite subset of points starting from the pairs and going to the bigger subsets, by making sure the constraints are satisfied as we are proceeding. <br> 
Since checking for subadditivity on overlapping subsets is very expensive, this code generate a random diversity of up to 7 nodes in a reasonable amount of time. <br>
You could also randomly generate all the numbers first and check if its a diversity or not, but it is a lot more expensive than incremental generation.

The following function finds the subsets of size m from set S

In [3]:
import itertools
def findsubsets(S,m):
    return [frozenset(c) for c in itertools.combinations(S, m)]

The following function checks if the dictionary of (nodes) -> weights, are a diversity or not by checking if the diversity axioms are satisfied. <br>
It checks for the alternate axioms of monotonicity and subaddivity on overlapping sets.

In [12]:
def is_diversity(edges: dict):
    """
    edges: dict mapping frozenset(nodes) -> weight
    """
    for A, wA in edges.items():
        for B, wB in edges.items():
            if A == B:
                continue

            # 1. Subset monotonicity
            if A.issubset(B):
                if wA > wB:
                    return False, f"Violation of property 1: {A} <= {B} but {wA} > {wB}"

            # 2. Subadditivity on overlapping sets
            if A & B:  # non-empty intersection
                union = A | B
                if union in edges:  # only check if union edge exists
                    if edges[union] > wA + wB:
                        return False, f"Violation of property 2: {union} weight {edges[union]} > {wA}+{wB}"

    return True, "All constraints satisfied"

In [16]:
import numpy as np
import random
def generate_random_diversity(n: int, max_weight: int = 100, max_weight_pairs: int = -1) -> dict:
    """
    n : int
        Number of total objects in the diversity space.
    max_weight : int, optional
        Maximum weight of all the edges (default 100).
    max_weight_pairs : int, optional
        Maximum weight allowed for pairs of nodes.
        Defaults to max_weight // 2.
    """
    nodes = np.arange(n)
    if max_weight_pairs == -1:
        max_weight_pairs = max_weight // 2

    while True:  # retry loop
        all_edges = {}
        success = True

        for size in range(2, n + 1):
            for S in findsubsets(nodes, size):

                if size == 2:  # initializing pairs
                    all_edges[S] = random.randint(1, max_weight_pairs)
                    continue

                # monotonicity lower bound
                lower_bound = monotonicity_lower_bound(S, size, all_edges)
                if lower_bound > max_weight:
                    success = False
                    break

                # subadditivity upper bound
                upper_bound = subadditivity_upper_bound(S, size, all_edges)
                upper_bound = min(upper_bound, max_weight)

                if lower_bound > upper_bound:
                    success = False
                    break

                all_edges[S] = random.randint(lower_bound, upper_bound)

            if not success:
                break  # break outer size loop too

        if success:
            return all_edges

def monotonicity_lower_bound(S: set, size: int, edges: dict) -> int:
    lower_bound = max([edges[subset] for subset in findsubsets(S, size - 1)] + [1])
    return lower_bound
    
def subadditivity_upper_bound(S: set, size: int, edges: dict) -> int:
    
    if size % 2 == 0:
        a, b = size // 2 + 1, size // 2
    else:
        a = b = (size + 1) // 2

    upper_bound_candidates = []
    for i in range(2, b + 1):
        for A in findsubsets(S, size + 1 - i):
            D = S - A
            for extra in findsubsets(A, 1):
                B = D | extra
                upper_bound_candidates.append(edges[A] + edges[B])
    upper_bound = min(upper_bound_candidates)
    
    return upper_bound

In [20]:
edges = generate_random_diversity(6)
a = is_diversity(edges)
print(a)

(True, 'All constraints satisfied')
